# Docker Basics Interactive Workshop (Jupyter)

Welcome! This notebook walks through Docker fundamentals step by step.

Open this file in **VS Code** with the [Jupyter extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) or in **Jupyter Lab**, then run each cell in order.

**Estimated time:** 30–45 minutes

> **Note:** Interactive shells (`docker exec -it`) work best in a terminal. Module 2 includes a terminal alternative for that step.

---

## Module 0: Prerequisites

Make sure Docker Engine is installed before continuing.

In [ ]:
%%bash
docker --version
docker info --format '{{.ServerVersion}}' 2>/dev/null || docker info | head -5


---

## Module 1: Creating an Image

A **Dockerfile** describes how to build an image. Inspect `examples/hello-docker/` — it uses nginx to serve a simple HTML page.

In [ ]:
%cd examples/hello-docker
%%bash
docker build -t hello-docker:1.0 .


The `-t` flag **tags** the image with a name (`hello-docker`) and version (`1.0`). Docker caches each instruction as a **layer** to speed up rebuilds.

In [ ]:
%%bash
docker images hello-docker


---

## Module 2: Docker Run and Exec

`docker run` creates and starts a container. `docker exec` runs a command inside a running container.

In [ ]:
%%bash
docker run -d --name hello -p 8080:80 hello-docker:1.0


- `-d` — detached (background)
- `--name hello` — friendly container name
- `-p 8080:80` — map host port 8080 to container port 80

In [ ]:
%%bash
docker ps


In [ ]:
%%bash
docker ps -a


`docker ps` shows only **running** containers. `-a` includes stopped ones.

In [ ]:
%%bash
curl -s localhost:8080


In [ ]:
%%bash
docker exec hello cat /usr/share/nginx/html/index.html


### Interactive shell (run in terminal)

Jupyter cannot attach a TTY the way a terminal does. Open a terminal and run:

```bash
docker exec -it hello sh
```

Try `hostname` or `ls /usr/share/nginx/html`, then type `exit`.

In [ ]:
%%bash
docker logs hello


---

## Module 3: Commit an Image

`docker commit` saves a container's filesystem state as a **new image**. In production, prefer updating the Dockerfile and rebuilding.

In [ ]:
%%bash
docker exec hello sh -c 'echo "committed-layer" > /tmp/marker'


In [ ]:
%%bash
docker commit hello hello-docker:committed


In [ ]:
%%bash
docker images hello-docker


In [ ]:
%%bash
docker run --rm hello-docker:committed cat /tmp/marker


The marker file exists in the committed image even though it was never in the original Dockerfile.

---

## Module 4: Container and Image Management

In [ ]:
%%bash
docker stop hello


In [ ]:
%%bash
docker ps -a --filter name=hello


In [ ]:
%%bash
docker start hello


In [ ]:
%%bash
docker ps --filter name=hello


### Remove the container

Run when ready — permanently removes `hello`.

In [ ]:
%%bash
docker rm -f hello


### Remove an image

Run after the container is removed.

In [ ]:
%%bash
docker rmi hello-docker:1.0


In [ ]:
%%bash
docker image ls hello-docker


### Prune unused images (optional)

In [ ]:
%%bash
docker image prune -f


---

## Module 5: Docker Compose with Bridge Networks

Custom **bridge** networks isolate services while allowing DNS-based discovery.

- **web** (nginx) on `frontend` — port 8081
- **api** (Python) on `frontend` + `backend`
- **redis** on `backend` only

In [ ]:
%cd examples/compose-demo
%%bash
docker compose config


In [ ]:
%%bash
docker network ls


In [ ]:
%cd examples/compose-demo
%%bash
docker compose up -d --build --wait


`--wait` blocks until health checks pass.

In [ ]:
%%bash
docker network ls --filter name=compose-demo


In [ ]:
%%bash
docker network inspect compose-demo_frontend --format '{{.Driver}}: {{range .Containers}}{{.Name}} {{end}}'


In [ ]:
%%bash
curl -s localhost:8081


In [ ]:
%%bash
curl -s localhost:8081/api/health


Run the API cell a few times — `redis_hits` should increment.

In [ ]:
%cd examples/compose-demo
%%bash
docker compose ps


### Tear down compose stack

In [ ]:
%cd examples/compose-demo
%%bash
docker compose down --volumes --remove-orphans


---

## Module 6: Final Cleanup

In [ ]:
%%bash
docker rm -f hello 2>/dev/null || true
docker compose -f examples/compose-demo/docker-compose.yml down --volumes --remove-orphans 2>/dev/null || true


In [ ]:
%%bash
docker rmi hello-docker:1.0 hello-docker:committed 2>/dev/null || true
docker rmi compose-demo-web compose-demo-api 2>/dev/null || true


In [ ]:
%%bash
echo "Remaining workshop containers:"
docker ps -a --filter name=hello --filter name=compose-demo --format '{{.Names}}' | grep -E 'hello|compose-demo' || echo "(none)"
echo "Remaining workshop images:"
docker images --format '{{.Repository}}:{{.Tag}}' | grep -E 'hello-docker|compose-demo' || echo "(none)"


---

## Congratulations!

You covered:
1. **Building** an image from a Dockerfile
2. **Running** containers and **exec** into them
3. **Committing** container changes to a new image
4. **Managing** containers and images
5. **Docker Compose** with custom **bridge** networks

**Challenge:** Modify `examples/hello-docker/app/index.html`, rebuild, and run a new container.